# Introduction exercises

**Authors:** Alja Dostal, Živa Artnak, Manca Kavčič, Jure Snoj

In [1]:
from pathlib import Path

In [2]:
# Set (local) data directory
project_root = Path('.') / '..'
data_dir = project_root / 'notebooks' / 'data'

## Data wrangling: `big-claim-events.csv` (*)

Load into your favorite data frame (R, pandas, polars, ...) the data [data/big-claim-events.csv](data/big-claim-events.csv), and for each of the four sub-populations defined by $\textrm{skin-cancer}=i, \textrm{depression}=j$, where $i,j \in \{0, 1\}$, calculate the ratio of big-claims to total claims for the sub-population.

### Solution

In [3]:
import pandas as pd

big_claim_df = pd.read_csv(data_dir / 'big-claim-events.csv')

# big_claim is 0/1, so summing it gives the number of big claims and the group
# size gives the total claims -> the ratio is just the mean within each subgroup.
ratios = (
    big_claim_df.groupby(['skin_cancer', 'depression'])['big_claim']
    .agg(big_claims='sum', total='size')
)
ratios['ratio'] = ratios['big_claims'] / ratios['total']
ratios

big_claims  total     ratio
skin_cancer depression                             
0.0         0.0               1192   1563  0.762636
            1.0                384    468  0.820513
1.0         0.0                297    363  0.818182
            1.0                 95    114  0.833333

## Data quality: `aggregate-claim.csv`

Load into some data frame [../notebooks/data/aggregate-claim.csv](../notebooks/data/aggregate-claim.csv). Identify at least one potential data quality issue.

Hint: There are some convenient data profiling tools for both python (e.g. [fg-data-profiling](https://github.com/Data-Centric-AI-Community/fg-data-profiling), formerly `ydata-profiling`, formerly-er `pandas-profiling`) and R (e.g. [DataExplorer](https://cran.r-project.org/web/packages/DataExplorer/vignettes/dataexplorer-intro.html)).

### Solution

In [4]:
import pandas as pd
from data_profiling import ProfileReport

In [5]:
aggregate_claim_df = pd.read_csv(data_dir / 'aggregate-claim.csv')
aggregate_claim_df.head()

,customer_id,agg_claim_amount,year
0,0,28848.260,2022
1,1,34775.650,2022
2,2,16875.703,2022
3,3,27648.465,2022
4,4,20385.668,2022


In [6]:
# Expect 836 customers x 3 years (2022-2024) = 2508 rows, but there are more.
print('rows:', len(aggregate_claim_df))
print('exact duplicate rows:', aggregate_claim_df.duplicated().sum())
print(aggregate_claim_df['year'].value_counts().sort_index())

# e.g. customer 0 has their 2023 row twice
aggregate_claim_df[aggregate_claim_df['customer_id'] == 0]

rows: 3344
exact duplicate rows: 836
year
2022     836
2023    1672
2024     836
Name: count, dtype: int64


,customer_id,agg_claim_amount,year
0,0,28848.260,2022
836,0,24179.393,2023
1672,0,24179.393,2023
2508,0,29326.541,2024


The problem is duplicate records. There are 836 exact duplicate rows: every customer's 2023 claim is in the file twice, so 2023 has 1672 rows against 836 for 2022 and 2024, and the file has 3344 rows instead of the expected 836 × 3 = 2508. If you left them in, anything aggregated by year would double-count 2023. The de-duplicated version is `aggregate-claim-1.csv`, which is the one used later in the high-risk notebook.